# Тропин Никита
## **Домашнее задание №14**

### Тема: эмбеддинги, индекс `FAISS`, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

##### **1. Импорты, seed и среда**

In [97]:
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Создаем папку для артефактов
os.makedirs("artifacts", exist_ok=True)

In [98]:
# Загрузка FAISS
def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False

FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")
if FAISS_READY:
    import faiss  # type: ignore

SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

In [99]:
# Фиксация seed
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [100]:
# Устройство
try:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

In [101]:
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Устройство: {DEVICE}")
print(f"FAISS доступен: {FAISS_READY}")
print(f"Sentence-transformers доступен: {SENTENCE_TRANSFORMERS_READY}")

NumPy: 2.0.2
Pandas: 2.2.2
Устройство: cuda
FAISS доступен: True
Sentence-transformers доступен: True


##### **2. База знаний и первичный анализ**

В качестве базы знаний выбрана подборка из 10 коротких статей об объектах Солнечной системы.

**Предметная область:** `астрономия`.

Данная тема отлично подходит для задачи RAG, так как содержит конкретные факты, цифры и термины, по которым удобно проверять качество извлечения контекста.

In [102]:
documents: List[Dict[str, str]] = [
    {"doc_id": "doc_01", "title": "Солнечная система", "text": "Солнечная система — планетная система, включающая в себя центральную звезду Солнце и все естественные космические объекты, вращающиеся вокруг неё. Она сформировалась путём гравитационного сжатия газопылевого облака около 4,57 млрд лет назад."},
    {"doc_id": "doc_02", "title": "Солнце", "text": "Солнце — единственная звезда Солнечной системы. Вокруг неё обращаются другие объекты этой системы. Масса Солнца составляет 99,86 % от суммарной массы всей Солнечной системы. Солнце состоит из водорода и гелия."},
    {"doc_id": "doc_03", "title": "Меркурий", "text": "Меркурий — самая маленькая планета Солнечной системы и самая близкая к Солнцу. Названа в честь древнеримского бога торговли. Поверхность Меркурия сильно кратерирована и напоминает лунную."},
    {"doc_id": "doc_04", "title": "Венера", "text": "Венера — вторая по удалённости от Солнца планета. Атмосфера Венеры состоит в основном из углекислого газа и азота, а давление на поверхности в 92 раза больше земного, что создает мощнейший парниковый эффект."},
    {"doc_id": "doc_05", "title": "Земля и Луна", "text": "Земля — третья планета от Солнца, единственное известное человеку тело во Вселенной, населённое живыми организмами. Луна — единственный естественный спутник Земли. Луна покрыта кратерами и морями из застывшей базальтовой лавы."},
    {"doc_id": "doc_06", "title": "Марс", "text": "Марс — четвёртая планета Солнечной системы. Названа в честь древнеримского бога войны из-за своего красного цвета, который обусловлен оксидом железа. На Марсе находится Олимп — высочайшая гора в Солнечной системе."},
    {"doc_id": "doc_07", "title": "Юпитер", "text": "Юпитер — крупнейшая планета Солнечной системы, пятая по удалённости от Солнца. Это газовый гигант, масса которого в два с половиной раза превышает суммарную массу всех остальных планет. Одной из главных его деталей является Большое красное пятно."},
    {"doc_id": "doc_08", "title": "Сатурн", "text": "Сатурн — шестая планета от Солнца и вторая по размерам после Юпитера. Сатурн известен своей заметной системой колец, состоящих главным образом из частичек льда, меньшего количества тяжёлых элементов и пыли."},
    {"doc_id": "doc_09", "title": "Уран и Нептун", "text": "Уран и Нептун — ледяные гиганты Солнечной системы. Уран уникален тем, что вращается вокруг Солнца «лежа на боку». Нептун — самая дальняя планета, известная сильными ветрами и Большим тёмным пятном."},
    {"doc_id": "doc_10", "title": "Пояс астероидов", "text": "Пояс астероидов — область Солнечной системы, расположенная между орбитами Марса и Юпитера, являющаяся местом скопления множества объектов всевозможных размеров, преимущественно неправильной формы, называемых астероидами."},
]

docs_df = pd.DataFrame(documents)
print("Число документов:", len(docs_df))
display(docs_df[["doc_id", "title"]].head())

Число документов: 10


,doc_id,title
0,doc_01,Солнечная система
1,doc_02,Солнце
2,doc_03,Меркурий
3,doc_04,Венера
4,doc_05,Земля и Луна


##### **3. Чанкинг документов**

Текст разбивается на фрагменты (чанки) по количеству слов.

`chunk_size` определяет максимальное количество слов в чанке,

а `overlap` — пересечение слов между соседними чанками, чтобы не разрывать контекст.

In [103]:
def chunk_text(text: str, chunk_size: int = 25, overlap: int = 5) -> List[str]:
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

In [104]:
# Демонстрация чанкинга:
demo_chunks = chunk_text(documents[0]["text"], chunk_size=15, overlap=3)
print(f"Оригинальный текст: {documents[0]['text']}")
for i, c in enumerate(demo_chunks):
    print(f"Чанк {i+1}: {c}")

Оригинальный текст: Солнечная система — планетная система, включающая в себя центральную звезду Солнце и все естественные космические объекты, вращающиеся вокруг неё. Она сформировалась путём гравитационного сжатия газопылевого облака около 4,57 млрд лет назад.
Чанк 1: Солнечная система — планетная система, включающая в себя центральную звезду Солнце и все естественные космические
Чанк 2: все естественные космические объекты, вращающиеся вокруг неё. Она сформировалась путём гравитационного сжатия газопылевого облака около
Чанк 3: газопылевого облака около 4,57 млрд лет назад.


##### **4. Эмбеддинги и индекс `FAISS`**

In [105]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

In [106]:
class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(texts, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        return self.fit_documents(texts)

In [107]:
class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

In [108]:
def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device)
        except Exception as e:
            print("Ошибка загрузки SentenceTransformers, используем TF-IDF:", e)
    return TfidfFallbackBackend()

In [109]:
@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object

In [110]:
def build_retriever(docs: List[Dict[str, str]], chunk_size: int = 25, overlap: int = 5, device: str = "cpu") -> RetrieverArtifacts:
    rows = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for i, chunk_txt in enumerate(parts):
            rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": f"{doc['doc_id']}_{i}", "chunk_text": chunk_txt})

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if FAISS_READY:
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
    else:
        index = vectors # Используем саму матрицу для fallback

    return RetrieverArtifacts(backend.backend_name, chunks_df, vectors, backend, index)

In [111]:
def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    q_vec = artifacts.backend.encode_queries([query]).astype(np.float32)
    if FAISS_READY:
        scores, indices = artifacts.index.search(q_vec, top_k)
        scores, indices = scores[0], indices[0]
    else:
        sim = (artifacts.chunk_vectors @ q_vec.T).reshape(-1)
        indices = np.argsort(-sim)[:top_k]
        scores = sim[indices]

    res = []
    for rank, (score, idx) in enumerate(zip(scores, indices), 1):
        row = artifacts.chunks_df.iloc[int(idx)]
        res.append({"rank": rank, "score": float(score), "doc_id": row["doc_id"], "title": row["title"], "chunk_text": row["chunk_text"]})
    return pd.DataFrame(res)

In [112]:
# Построение baseline retriever
base_artifacts = build_retriever(documents, chunk_size=25, overlap=5, device=DEVICE)
print(f"Backend: {base_artifacts.backend_name}")
print(f"Всего чанков: {len(base_artifacts.chunks_df)}")

# Тестовый поиск
display(search_chunks("Из чего состоит атмосфера Венеры?", base_artifacts, top_k=2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Всего чанков: 18


,rank,score,doc_id,title,chunk_text
0,1,0.711766,doc_04,Венера,Венера — вторая по удалённости от Солнца плане...
1,2,0.430246,doc_01,Солнечная система,"Солнечная система — планетная система, включаю..."


##### **5. Контрольные запросы и оценка retrieval**

Было сформировано 10 тестовых запросов и проверим метрики hit@k и recall@k.

In [113]:
benchmark_queries = [
    {"query_id": "q01", "query": "Возраст формирования Солнечной системы", "relevant_doc_ids": ["doc_01"]},
    {"query_id": "q02", "query": "Из каких газов состоит звезда Солнце?", "relevant_doc_ids": ["doc_02"]},
    {"query_id": "q03", "query": "Какая планета самая близкая к Солнцу?", "relevant_doc_ids": ["doc_03"]},
    {"query_id": "q04", "query": "Где самое высокое давление на поверхности?", "relevant_doc_ids": ["doc_04"]},
    {"query_id": "q05", "query": "Какой спутник вращается вокруг Земли?", "relevant_doc_ids": ["doc_05"]},
    {"query_id": "q06", "query": "Почему Марс имеет красный цвет?", "relevant_doc_ids": ["doc_06"]},
    {"query_id": "q07", "query": "Где находится Большое красное пятно?", "relevant_doc_ids": ["doc_07"]},
    {"query_id": "q08", "query": "Планета с заметной системой колец изо льда", "relevant_doc_ids": ["doc_08"]},
    {"query_id": "q09", "query": "Какая планета вращается лежа на боку?", "relevant_doc_ids": ["doc_09"]},
    {"query_id": "q10", "query": "Где находится пояс астероидов?", "relevant_doc_ids": ["doc_10"]},
]

In [114]:
def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered

In [115]:
def evaluate_query(query: str, relevant_doc_ids: List[str], artifacts: RetrieverArtifacts, top_k: int = 3) -> Dict:
    res_df = search_chunks(query, artifacts, top_k=top_k)
    pred_docs = unique_doc_order(res_df)

    hit = int(any(d in pred_docs for d in relevant_doc_ids))
    recall = sum(d in pred_docs for d in relevant_doc_ids) / len(relevant_doc_ids)

    first_rank = next((idx for idx, d in enumerate(pred_docs, 1) if d in relevant_doc_ids), None)
    mrr = 0.0 if first_rank is None else 1.0 / first_rank

    return {"predicted_doc_ids": pred_docs, "hit": hit, "recall": recall, "mrr": mrr, "first_relevant_rank": first_rank}

In [116]:
def evaluate_benchmark(queries: List[Dict], artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    rows = []
    for q in queries:
        eval_res = evaluate_query(q["query"], q["relevant_doc_ids"], artifacts, top_k)
        rows.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "expected_source": ", ".join(q["relevant_doc_ids"]),
            "retrieved_sources": ", ".join(eval_res["predicted_doc_ids"]),
            f"hit_at_k": eval_res["hit"],
            f"recall_at_k": eval_res["recall"],
            f"mrr_at_k": eval_res["mrr"],
            "rank_of_first_relevant": eval_res["first_relevant_rank"]
        })
    return pd.DataFrame(rows)

In [117]:
base_eval_df = evaluate_benchmark(benchmark_queries, base_artifacts, top_k=3)
display(base_eval_df.head(5))

# Сохранение артефакта
base_eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,mrr_at_k,rank_of_first_relevant
0,q01,Возраст формирования Солнечной системы,doc_01,"doc_01, doc_02",1,1.0,1.0,1.0
1,q02,Из каких газов состоит звезда Солнце?,doc_02,"doc_02, doc_01",1,1.0,1.0,1.0
2,q03,Какая планета самая близкая к Солнцу?,doc_03,"doc_09, doc_02",0,0.0,0.0,NaN
3,q04,Где самое высокое давление на поверхности?,doc_04,"doc_04, doc_08, doc_07",1,1.0,1.0,1.0
4,q05,Какой спутник вращается вокруг Земли?,doc_05,"doc_05, doc_01, doc_09",1,1.0,1.0,1.0


##### **6. Небольшой эксперимент с параметрами retrieval**

Сравним `chunk_size = 15` и `chunk_size = 40` при фиксированном `overlap = 5`.

In [118]:
exp_configs = [
    {"chunk_size": 15, "overlap": 5},
    {"chunk_size": 40, "overlap": 5}
]

exp_results = []
for cfg in exp_configs:
    exp_art = build_retriever(documents, chunk_size=cfg["chunk_size"], overlap=cfg["overlap"], device=DEVICE)
    eval_df = evaluate_benchmark(benchmark_queries, exp_art, top_k=3)
    exp_results.append({
        "chunk_size": cfg["chunk_size"],
        "num_chunks": len(exp_art.chunks_df),
        "mean_hit@3": eval_df["hit_at_k"].mean(),
        "mean_MRR@3": eval_df["mrr_at_k"].mean()
    })

display(pd.DataFrame(exp_results))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_size,num_chunks,mean_hit@3,mean_MRR@3
0,15,29,1.0,0.883333
1,40,10,0.9,0.850000


**Вывод:**

Изменение размера чанка влияет на количество фрагментов.

В маленьком корпусе метрики могут оставаться высокими (1.0), однако на больших корпусах слишком мелкий чанк теряет контекст, а крупный захватывает шум.

##### **7. Обновление базы знаний и переиндексация**

Добавил 3 новых документа: про экзопланеты, черные дыры и телескоп "Джеймс Уэбб".

Надо посмотреть, как изменятся результаты на новых запросах до и после переиндексации.

In [119]:
new_documents = [
    {"doc_id": "doc_11", "title": "Экзопланеты", "text": "Экзопланета — планета, находящаяся вне Солнечной системы. Первые достоверно подтверждённые открытия экзопланет были сделаны в начале 1990-х годов. Поиск экзопланет активно ведётся с помощью транзитного метода."},
    {"doc_id": "doc_12", "title": "Чёрная дыра", "text": "Чёрная дыра — область пространства-времени, гравитационное притяжение которой настолько велико, что покинуть её не могут даже кванты самого света. Граница этой области называется горизонтом событий."},
    {"doc_id": "doc_13", "title": "Джеймс Уэбб", "text": "Космический телескоп «Джеймс Уэбб» — орбитальная инфракрасная обсерватория, запущенная в 2021 году. Этот телескоп предназначен для изучения формирования первых галактик и поиска экзопланет с условиями для жизни."}
]

updated_documents = documents + new_documents
updated_artifacts = build_retriever(updated_documents, chunk_size=25, overlap=5, device=DEVICE)

update_queries = [
    {"query_id": "q11", "query": "Как работает транзитный метод поиска?", "relevant_doc_ids": ["doc_11"]},
    {"query_id": "q12", "query": "Что такое горизонт событий?", "relevant_doc_ids": ["doc_12"]},
    {"query_id": "q13", "query": "Какой телескоп запущен в 2021 году?", "relevant_doc_ids": ["doc_13"]}
]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [120]:
# Оценка до обновления (на базе старого индекса base_artifacts)
eval_before = evaluate_benchmark(update_queries, base_artifacts, top_k=3)

# Оценка после обновления (на обновленном индексе updated_artifacts)
eval_after = evaluate_benchmark(update_queries, updated_artifacts, top_k=3)

compare_df = pd.merge(
    eval_before[["query", "expected_source", "retrieved_sources"]],
    eval_after[["query", "retrieved_sources"]],
    on="query", suffixes=("_before", "_after")
)
compare_df["changed"] = compare_df["retrieved_sources_before"] != compare_df["retrieved_sources_after"]

display(compare_df)

# Сохранение артефакта
compare_df.rename(columns={
    "retrieved_sources_before": "before_retrieved_sources",
    "retrieved_sources_after": "after_retrieved_sources"
}).to_csv("artifacts/retrieval_before_after_update.csv", index=False)

,query,expected_source,retrieved_sources_before,retrieved_sources_after,changed
0,Как работает транзитный метод поиска?,doc_11,"doc_05, doc_10","doc_11, doc_13",True
1,Что такое горизонт событий?,doc_12,"doc_09, doc_01","doc_12, doc_09, doc_13",True
2,Какой телескоп запущен в 2021 году?,doc_13,"doc_09, doc_01","doc_13, doc_11, doc_09",True


##### **8. Mini-RAG**

Надо построить конвейер, который извлекает текст, формирует контекст и с помощью механизма извлечения `(extractive summaries)` отвечает на вопрос.

*Примечание: Чтобы не усложнять зависимости LLM, генератор формирует ответ из наиболее релевантных предложений из найденного контекста.*

In [121]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

In [122]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []
    for _, row in retrieved.iterrows():
        block = f"[Doc: {row['doc_id']}] {row['chunk_text']}"
        context_blocks.append(block)
    return "\n".join(context_blocks), retrieved

In [123]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    # Удаляем служебные теги для чистого текста
    content_lines = [re.sub(r"\[Doc: .*?\]\s*", "", line) for line in raw_lines]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 3]
    if not sentence_pool:
        return "Недостаточно контекста для ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]
    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used = set()

    for idx in ranked_idx:
        if scores[idx] <= 0: continue
        sent = sentence_pool[idx]
        norm_sent = sent.lower()
        if norm_sent not in used:
            used.add(norm_sent)
            selected_sentences.append(sent)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В контексте нет ответа."
    return " ".join(selected_sentences)

In [124]:
def mini_rag_answer(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> dict:
    context, retrieved = build_context_from_retrieval(query, artifacts, top_k)
    answer = generate_answer_from_context(query, context)
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(retrieved["doc_id"].unique())
    }

In [125]:
# Демонстрация Mini-RAG:
rag_queries = [
    "Какой газ вызывает парниковый эффект на Венере?",
    "Чем известна планета Сатурн?",
    "В каком году запущен Джеймс Уэбб?",
    "Сколько лет назад сформировалась Солнечная система?",
    "Что такое Большое красное пятно?"
]

rag_results = []
for q in rag_queries:
    res = mini_rag_answer(q, updated_artifacts)
    rag_results.append(res)
    print(f"Q: {res['question']}\nA: {res['answer']}\nSources: {res['retrieved_sources']}\n{'-'*50}")

rag_df = pd.DataFrame(rag_results)

# Сохранение артефакта
rag_df.to_csv("artifacts/rag_examples.csv", index=False)

Q: Какой газ вызывает парниковый эффект на Венере?
A: на поверхности в 92 раза больше земного, что создает мощнейший парниковый эффект. Атмосфера Венеры состоит в основном из углекислого газа и азота, а давление на поверхности в 92 раза
Sources: doc_04, doc_07
--------------------------------------------------
Q: Чем известна планета Сатурн?
A: Сатурн — шестая планета от Солнца и вторая по размерам после Юпитера. Марс — четвёртая планета Солнечной системы.
Sources: doc_08, doc_06, doc_09
--------------------------------------------------
Q: В каком году запущен Джеймс Уэбб?
A: Космический телескоп «Джеймс Уэбб» — орбитальная инфракрасная обсерватория, запущенная в 2021 году.
Sources: doc_13, doc_01, doc_11
--------------------------------------------------
Q: Сколько лет назад сформировалась Солнечная система?
A: сформировалась путём гравитационного сжатия газопылевого облака около 4,57 млрд лет назад. Солнечная система — планетная система, включающая в себя центральную звезду Солнце и

##### **9. Краткий анализ ошибок**

##### **Результаты:**

###### **Успешные кейсы:**
Система корректно ответила на вопрос про `парниковый эффект на Венере`, `Джеймс Уэбб` и `формирование Солнечной системы`, опираясь на контекст, потому что лексика в запросе хорошо пересекается с фактами в тексте *(TF-IDF суммаризатор легко находит совпадения)*.

###### **Проблемные/Пограничные кейсы:**

*Запрос: Чем известна планета Сатурн?*

- Ответ вышел хорошим, но система ошибочно вернула определение Марса как дополнение.

*Запрос: Что такое Большое красное пятно?*

- Ответ вышел сжатым, а Mini-RAG добавил бессмысленное объяснение, что такое "Уран".

Такие "неуверенные" / "ошибочные" ответы на запросы могут происходить по двум причинам:

1. Ограничение генератора: TF-IDF алгоритм выбирает предложение, где больше всего пересечений токенов с вопросом, но ответ может логически находиться в *предыдущем* или *следующем* предложении. Настоящая LLM исправила бы эту проблему, "прочитав" весь контекст.
2. Неполнота базы: напрмиер, в тексте про `Большое красное пятно` не описывается физика пятна, лишь упоминается факт его наличия на Юпитере. RAG не может дать больше информации, чем содержится в самих документах.